# Libraries and Data Import

We use data downloaded from kaagle:

https://www.kaggle.com/datasets/urvishahir/electric-vehicle-specifications-dataset-2025

In [1]:
import json
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

/home/danny/Documentos/4-Curso/TFG/TFG-Server/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
file_path = "electric_vehicles_spec_2025.csv.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "urvishahir/electric-vehicle-specifications-dataset-2025",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

df.head()

,brand,model,top_speed_kmh,battery_capacity_kWh,battery_type,number_of_cells,torque_nm,efficiency_wh_per_km,range_km,acceleration_0_100_s,...,towing_capacity_kg,cargo_volume_l,seats,drivetrain,segment,length_mm,width_mm,height_mm,car_body_type,source_url
0,Abarth,500e Convertible,155,37.8,Lithium-ion,192.0,235.0,156,225,7.0,...,0.0,185,4,FWD,B - Compact,3673,1683,1518,Hatchback,https://ev-database.org/car/1904/Abarth-500e-C...
1,Abarth,500e Hatchback,155,37.8,Lithium-ion,192.0,235.0,149,225,7.0,...,0.0,185,4,FWD,B - Compact,3673,1683,1518,Hatchback,https://ev-database.org/car/1903/Abarth-500e-H...
2,Abarth,600e Scorpionissima,200,50.8,Lithium-ion,102.0,345.0,158,280,5.9,...,0.0,360,5,FWD,JB - Compact,4187,1779,1557,SUV,https://ev-database.org/car/3057/Abarth-600e-S...
3,Abarth,600e Turismo,200,50.8,Lithium-ion,102.0,345.0,158,280,6.2,...,0.0,360,5,FWD,JB - Compact,4187,1779,1557,SUV,https://ev-database.org/car/3056/Abarth-600e-T...
4,Aiways,U5,150,60.0,Lithium-ion,NaN,310.0,156,315,7.5,...,NaN,496,5,FWD,JC - Medium,4680,1865,1700,SUV,https://ev-database.org/car/1678/Aiways-U5


# Data Analysis and Cleaning

In [3]:
df.keys()

Index(['brand', 'model', 'top_speed_kmh', 'battery_capacity_kWh',
       'battery_type', 'number_of_cells', 'torque_nm', 'efficiency_wh_per_km',
       'range_km', 'acceleration_0_100_s', 'fast_charging_power_kw_dc',
       'fast_charge_port', 'towing_capacity_kg', 'cargo_volume_l', 'seats',
       'drivetrain', 'segment', 'length_mm', 'width_mm', 'height_mm',
       'car_body_type', 'source_url'],
      dtype='str')

We will use only "battery_capacity_kWh", "efficiency_wh_per_km" and "fast_charging_power_kw_dc".

In [4]:
df = df[["battery_capacity_kWh", "efficiency_wh_per_km", "fast_charging_power_kw_dc"]]

In [5]:
null_counts = df.isna().sum()
print("Null values:")
print(null_counts)

df = df.dropna().reset_index(drop=True)

print("\nNull values after cleaning:")
print(df.isna().sum())

Null values:
battery_capacity_kWh         0
efficiency_wh_per_km         0
fast_charging_power_kw_dc    1
dtype: int64

Null values after cleaning:
battery_capacity_kWh         0
efficiency_wh_per_km         0
fast_charging_power_kw_dc    0
dtype: int64


In [6]:
df.describe()

,battery_capacity_kWh,efficiency_wh_per_km,fast_charging_power_kw_dc
count,477.000000,477.000000,477.000000
mean,74.115094,162.974843,125.008386
std,20.292372,34.318323,58.205012
min,21.300000,109.000000,29.000000
25%,60.000000,143.000000,80.000000
50%,76.500000,155.000000,113.000000
75%,90.600000,178.000000,150.000000
max,118.000000,370.000000,281.000000


In [7]:
json_data = df.to_dict(orient="records")

with open("electric_vehicles_spec_2025.json", "w", encoding="utf-8") as f:
    json.dump(json_data, f, ensure_ascii=False, indent=2)

print(f"JSON saved with {len(json_data)} records.")

JSON saved with 477 records.


# DataBase Population

In [1]:
import json
import pandas as pd
import sys
sys.path.append('../..')

from src.Persistance.db_broker import DBBroker

In [2]:
with open('electric_vehicles_spec_2025.json', 'r') as f:
    json_data = json.load(f)

n_vehicles = len(json_data)
print(f"Number of vehicles: {n_vehicles}")

Number of vehicles: 477


## Users

In [3]:
from faker import Faker

fake = Faker('es_ES')

def generate_phone():
    return f"+34 {fake.numerify(text='6########')}"

def generate_name():
    name = fake.first_name().lower().replace(' ', '_').replace('á', 'a').replace('é', 'e').replace('í', 'i').replace('ó', 'o').replace('ú', 'u')
    surname = fake.last_name().lower().replace(' ', '_').replace('á', 'a').replace('é', 'e').replace('í', 'i').replace('ó', 'o').replace('ú', 'u')

    return f"{name}_{surname}"

reserved_emails = {'daniel.sanchez55@alu.uclm.es'}
used_emails = set(reserved_emails)
users = []

while len(users) < (n_vehicles - 1):
    name = generate_name()
    email = f"{name}@{fake.free_email_domain()}"
    if email in used_emails:
        continue

    phone = generate_phone()
    users.append((name, email, phone, 15.00))
    used_emails.add(email)

print(users[:5])

[('perla_alberola', 'perla_alberola@hotmail.com', '+34 622059447', 15.0), ('casandra_fabregas', 'casandra_fabregas@yahoo.com', '+34 686729283', 15.0), ('dafne_casado', 'dafne_casado@hotmail.com', '+34 678014420', 15.0), ('bernardino_carrillo', 'bernardino_carrillo@gmail.com', '+34 687618681', 15.0), ('vanesa_mendez', 'vanesa_mendez@yahoo.com', '+34 651145294', 15.0)]


In [4]:
query_users = """
    INSERT INTO ev_users (username, email, phone, value_of_time)
    VALUES (?, ?, ?, ?)
"""

In [5]:
my_user = ('danny', 'daniel.sanchez55@alu.uclm.es', '+34 926295429', 15.00)

DBBroker().execute_write_query(query_users, my_user)

2026-05-13 19:20:37,800 - DBBroker - 38 - INFO: Attempting to initialize Database Connection Pool...
2026-05-13 19:20:37,824 - DBBroker - 49 - INFO: Database connection pool established successfully.


(0, 1)

In [6]:
batch_size = 100

for i in range(0, len(users), batch_size):
    batch = users[i:i+batch_size]
    DBBroker().execute_many(query_users, batch)

## Vehicles

In [7]:
def generate_plate():
    alphabet = "BCDFGHJKLMNPRSTVWXYZ"

    return fake.bothify(text='#### ???', letters=alphabet)

In [8]:
generate_plate()

'1527 BHS'

In [9]:
users_ids = DBBroker().execute_read_query("SELECT id FROM ev_users")
users_ids = [u["id"] for u in users_ids]
users_ids[:5]

[223, 204, 174, 221, 419]

In [10]:
plates = set()

for c in json_data:
    p = generate_plate()
    while p in plates:
        p = generate_plate()

    plates.add(p)

    c["plate"] = p
    c["user_id"] = fake.random_element(elements=users_ids)

In [11]:
query_vehicles = """
    INSERT INTO vehicles (plate, consumption_wh_km, capacity_kwh, max_kw_speed, user_id)
    VALUES (?, ?, ?, ?, ?)
"""

In [12]:
my_plate = "1234 BCD"

my_car = (my_plate, json_data[0]['efficiency_wh_per_km'], json_data[0]['battery_capacity_kWh'], json_data[0]['fast_charging_power_kw_dc'], 1)

DBBroker().execute_write_query(query_vehicles, my_car)

(0, None)

In [13]:
batch_size = 100

for batch in range(0, len(json_data), batch_size):
    batch_data = json_data[batch:batch+batch_size]
    values = [(c['plate'], c['efficiency_wh_per_km'], c['battery_capacity_kWh'], c['fast_charging_power_kw_dc'], c['user_id']) for c in batch_data]

    DBBroker().execute_many(query_vehicles, values)